# Create SciQuality continual-learning stage files

Create the six finalized chronological train/holdout CSV files. Every split is separated by a complete 24-hour NF-only gap.

In [1]:
from pathlib import Path
import pandas as pd

# Find the repository root.
ROOT = Path.cwd().resolve()
while not (ROOT / "data_labeling").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Repository root could not be found.")
    ROOT = ROOT.parent

INPUT = ROOT / "data_labeling/data_labels/sciquality/labels_sciquality_2010_2026_binary.csv"
OUTPUT = ROOT / "data_labeling/data_labels/sciquality/continual_stages_2010_2024"

In [2]:
# Load labels and extract each image timestamp.
data = pd.read_csv(INPUT)
text = data["label"].str.extract(r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})")[0]
data["timestamp"] = pd.to_datetime(text, format="%Y.%m.%d_%H.%M.%S")
data = data.sort_values("timestamp").reset_index(drop=True)

assert data["label"].is_unique
assert set(data["goes_class"].unique()) <= {0, 1}
print(f"Images loaded: {len(data):,}")

Images loaded: 128,117


In [3]:
# Final boundaries selected in the split-analysis notebook.
PERIODS = {
    "Stage1_train.csv": ("2010-12-06 07:00", "2017-01-06 03:00"),
    "Stage1_holdout.csv": ("2017-01-07 04:00", "2017-12-31 23:00"),
    "Stage2_train.csv": ("2018-01-02 00:00", "2023-03-08 22:00"),
    "Stage2_holdout.csv": ("2023-03-09 23:00", "2023-06-29 22:00"),
    "Stage3_train.csv": ("2023-07-01 00:00", "2024-05-24 20:00"),
    "Stage3_holdout.csv": ("2024-05-25 21:00", "2024-06-29 10:00"),
}
GAPS = {
    "Stage 1": ("2017-01-06 04:00", "2017-01-07 03:00"),
    "Stage 2": ("2023-03-08 23:00", "2023-03-09 22:00"),
    "Stage 3": ("2024-05-24 21:00", "2024-05-25 20:00"),
}

datasets = {name: data[data["timestamp"].between(start, end)].copy()
            for name, (start, end) in PERIODS.items()}

In [4]:
# Verify that every excluded internal gap has 24 NF images and no missing hour.
for stage, (start, end) in GAPS.items():
    gap = data[data["timestamp"].between(start, end)]
    expected = pd.date_range(start, end, freq="h")
    assert len(gap) == 24 and gap["goes_class"].sum() == 0
    assert expected.difference(gap["timestamp"]).empty

summary = []
for name, frame in datasets.items():
    fl = int(frame["goes_class"].sum())
    summary.append({"dataset": name, "samples": len(frame),
                    "NF": len(frame) - fl, "FL": fl,
                    "FL_percent": 100 * fl / len(frame)})
pd.DataFrame(summary)

,dataset,samples,NF,FL,FL_percent
0,Stage1_train.csv,46638,35307,11331,24.295639
1,Stage1_holdout.csv,8230,7780,450,5.467801
2,Stage2_train.csv,44125,40273,3852,8.729745
3,Stage2_holdout.csv,2651,1446,1205,45.454545
4,Stage3_train.csv,7798,3726,4072,52.218518
5,Stage3_holdout.csv,826,226,600,72.639225


In [5]:
# Write only the columns required by the training pipeline.
OUTPUT.mkdir(parents=True, exist_ok=True)
for name, frame in datasets.items():
    path = OUTPUT / name
    frame[["label", "goes_class"]].to_csv(path, index=False)
    print(f"Created: {path} ({len(frame):,} images)")

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/sciquality/continual_stages_2010_2024/Stage1_train.csv (46,638 images)
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/sciquality/continual_stages_2010_2024/Stage1_holdout.csv (8,230 images)
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/sciquality/continual_stages_2010_2024/Stage2_train.csv (44,125 images)
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/sciquality/continual_stages_2010_2024/Stage2_holdout.csv (2,651 images)
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/sciquality/continual_stages_2010_2024/Stage3_train.csv (7,798 images)
Created: /Users/piyushluitel/Desktop/PhD/Research/Full